# Phase 4: Threading Golden Dataset & Grid Search

## 목표
1. **Golden Dataset 구성**: 낮은 threshold로 over-thread → Gemini 검증 → ground truth 확보
2. **Evaluation Harness**: contamination / fragmentation / causal_recall 3-metric composite
3. **Grid Search**: 1,536 parameter combinations → best composite score 찾기

## 현재 상태 (2026-03-02)
- 102 threads, 348/2,118 articles threaded (16.4%)
- Phase 4.0: time-weighted centroid + IDF entity overlap + author boost (EMA/size penalty 제거)
- DB scope: `published_at < '2026-03-01'`
- Plan: `docs/workflow/2-prds/6-prd-threading-overhaul.md`

## 접근 방식
```
Cell 4: Low-threshold in-memory simulation → golden_baseline.json
Cell 5-7: Gemini Q1/Q2/Q3 validation
Cell 8: Manual review + golden_dataset.json
Cell 9: evaluate_threading() harness
Cell 10: Grid search (1,536 combos, no DB calls)
Cell 11: Results analysis + heatmaps
Cell 12: Best params → apply to scripts/7_embed_and_thread.py
```

In [1]:
# Cell 1: Setup
import gc, json, math, os, time, pickle, itertools
from pathlib import Path
from datetime import datetime, timedelta, timezone
from collections import defaultdict

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv(Path('..') / '.env.local')
url = os.getenv('NEXT_PUBLIC_SUPABASE_URL') or os.getenv('SUPABASE_URL')
key = os.getenv('SUPABASE_SERVICE_ROLE_KEY')
assert url and key, 'Missing Supabase credentials'
supabase = create_client(url, key)
print('Connected to Supabase')

Connected to Supabase


In [2]:
# Cell 2: Load Data

def paginate_table(table, select_cols, filters=None, order_col=None, page_size=1000):
    """Paginate a Supabase table query.

    Args:
        order_col: Column to order by (required for deterministic pagination).
                   If None, defaults to 'id' for safety.
    """
    all_rows = []
    offset = 0
    while True:
        q = supabase.table(table).select(select_cols)
        if filters:
            for col, op, val in filters:
                q = q.filter(col, op, val)
        q = q.order(order_col or 'id')
        batch = q.range(offset, offset + page_size - 1).execute().data or []
        all_rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return all_rows

SCOPE_DATE = '2026-03-01'  # Exclude post-merge articles

print('Loading wsj_story_threads...')
threads_rows = paginate_table(
    'wsj_story_threads',
    'id, title, centroid, member_count, status, first_seen, last_seen'
)
print(f'  threads: {len(threads_rows)}')

print('Loading wsj_items...')
items_rows = paginate_table(
    'wsj_items',
    'id, title, description, published_at, creator, thread_id',
    filters=[('published_at', 'lt', SCOPE_DATE)],
    order_col='published_at'
)
print(f'  items: {len(items_rows)}')

print('Loading wsj_embeddings (paginated)...')
embeddings_rows = paginate_table('wsj_embeddings', 'wsj_item_id, embedding')
print(f'  embeddings: {len(embeddings_rows)}')

print('Loading wsj_llm_analysis (via crawl join)...')
# Join: wsj_items → wsj_crawl_results → wsj_llm_analysis
llm_rows = paginate_table(
    'wsj_items',
    'id, wsj_crawl_results(wsj_llm_analysis(key_entities, people_mentioned, keywords, tickers_mentioned, summary))',
    filters=[('published_at', 'lt', SCOPE_DATE)],
    order_col='published_at'
)
print(f'  llm join rows: {len(llm_rows)}')

# Build lookup maps
embedding_map = {}  # item_id -> np.ndarray
for row in embeddings_rows:
    raw = row['embedding']
    if isinstance(raw, str):
        raw = json.loads(raw)
    if raw:
        embedding_map[row['wsj_item_id']] = np.array(raw, dtype=np.float32)

entity_map = {}    # item_id -> [entity_str]
summary_map = {}   # item_id -> best summary string
keywords_map = {}  # item_id -> [keyword strings]
for row in llm_rows:
    entities = []
    best_summary = ''
    best_keywords = []
    crawl_list = row.get('wsj_crawl_results') or []
    if not isinstance(crawl_list, list):
        crawl_list = [crawl_list]
    for cr in crawl_list:
        if not cr:
            continue
        analyses = cr.get('wsj_llm_analysis') or []
        if not isinstance(analyses, list):
            analyses = [analyses]
        for ll in analyses:
            if not ll:
                continue
            for field in ('key_entities', 'people_mentioned', 'keywords', 'tickers_mentioned'):
                vals = ll.get(field) or []
                if isinstance(vals, list):
                    entities.extend(vals)
            if ll.get('summary') and not best_summary:
                best_summary = ll['summary']
            if ll.get('keywords') and not best_keywords:
                kw = ll['keywords']
                if isinstance(kw, list):
                    best_keywords = kw
            break  # First analysis is sufficient
    entity_map[row['id']] = entities
    if best_summary:
        summary_map[row['id']] = best_summary
    if best_keywords:
        keywords_map[row['id']] = best_keywords

print('\nData loaded.')
print(f'  embedding_map: {len(embedding_map)} articles')
print(f'  entity_map: {len(entity_map)} articles')
print(f'  summary_map: {len(summary_map)} articles')
print(f'  keywords_map: {len(keywords_map)} articles')

# Validate: how many items have embeddings?
items_with_emb = sum(1 for r in items_rows if r['id'] in embedding_map)
items_without_emb = len(items_rows) - items_with_emb
print(f'  items with embeddings: {items_with_emb} ({items_with_emb/len(items_rows):.1%})')
if items_without_emb > 0:
    print(f'  items WITHOUT embeddings: {items_without_emb} (will be singletons)')

Loading wsj_story_threads...
  threads: 119
Loading wsj_items...
  items: 2102
Loading wsj_embeddings (paginated)...
  embeddings: 2220
Loading wsj_llm_analysis (via crawl join)...
  llm join rows: 2102

Data loaded.
  embedding_map: 2220 articles
  entity_map: 2102 articles
  summary_map: 1723 articles
  keywords_map: 1721 articles
  items with embeddings: 2102 (100.0%)


In [3]:
# Cell 3: Build Baseline Thread Snapshot

# Build system threads from current DB state
system_threads = defaultdict(list)   # thread_id -> [article_id]
article_thread_map = {}              # article_id -> thread_id
threads_meta = {}                    # thread_id -> {title, member_count, status}

for item in items_rows:
    tid = item.get('thread_id')
    if tid:
        system_threads[tid].append(item['id'])
        article_thread_map[item['id']] = tid

for t in threads_rows:
    threads_meta[t['id']] = {
        'title': t['title'],
        'member_count': t['member_count'],
        'status': t['status'],
        'first_seen': t.get('first_seen'),
        'last_seen': t.get('last_seen'),
    }

total_threads = len(system_threads)
threaded_articles = len(article_thread_map)
total_articles = len(items_rows)
threading_rate = threaded_articles / total_articles if total_articles else 0

print('=== Baseline Thread Snapshot ===')
print(f'Total threads:       {total_threads}')
print(f'Threaded articles:   {threaded_articles} / {total_articles} ({threading_rate:.1%})')

# Status distribution
status_dist = defaultdict(int)
for t in threads_rows:
    status_dist[t.get('status', 'unknown')] += 1
print('\nStatus distribution:')
for status, count in sorted(status_dist.items()):
    print(f'  {status}: {count}')

# Member count distribution
member_counts = [t['member_count'] for t in threads_rows if t.get('member_count')]
if member_counts:
    mc = np.array(member_counts)
    print(f'\nMember count distribution:')
    print(f'  min={mc.min()}, median={np.median(mc):.0f}, mean={mc.mean():.1f}, max={mc.max()}')
    bins = [1, 2, 5, 10, 20, 50, 200]
    for lo, hi in zip(bins, bins[1:]):
        cnt = np.sum((mc >= lo) & (mc < hi))
        print(f'  [{lo:3d}-{hi:3d}): {cnt}')

=== Baseline Thread Snapshot ===
Total threads:       102
Threaded articles:   374 / 2102 (17.8%)

Status distribution:
  active: 17
  archived: 95
  cooling: 7

Member count distribution:
  min=2, median=3, mean=3.8, max=17
  [  1-  2): 0
  [  2-  5): 87
  [  5- 10): 29
  [ 10- 20): 3
  [ 20- 50): 0
  [ 50-200): 0


In [4]:
# Cell 4: Golden Dataset — Baseline with Low Threshold
# In-memory simulation of Phase 4.0 match_to_threads() with lower thresholds
# to over-thread → more material for Gemini to validate.

# ── Constants for golden baseline (lower than production) ──
GS_BASE_THRESHOLD = 0.58
GS_AUTHOR_THRESHOLD = 0.45
GS_TIME_PENALTY = 0.01
GS_CENTROID_DECAY = 0.10
GS_ENTITY_WEIGHT = 0.04
GS_MATCH_MARGIN = 0.03
GS_HARD_CAP = 50
GS_FROZEN_THRESHOLD = 0.87
GS_AUTHOR_WINDOW_HOURS = 48

# ── Entity helpers (reimplemented from 7_embed_and_thread.py) ──

_TITLE_PREFIXES = {
    'president', 'secretary', 'ceo', 'cfo', 'coo', 'cto', 'dr', 'mr', 'ms',
    'mrs', 'sen', 'rep', 'gov', 'gen', 'col', 'lt', 'chairman', 'director',
}


def normalize_entities(entities):
    cleaned = []
    for e in (entities or []):
        if not e or not e.strip():
            continue
        words = e.strip().split()
        while words and words[0].rstrip('.').lower() in _TITLE_PREFIXES:
            words = words[1:]
        if words:
            cleaned.append(' '.join(words))
    cleaned_lower = [c.lower() for c in cleaned]
    kept = []
    for i, e_lower in enumerate(cleaned_lower):
        dominated = any(e_lower != other and e_lower in other for other in cleaned_lower)
        if not dominated:
            kept.append(cleaned[i])
    seen = set()
    result = []
    for e in kept:
        key = e.lower()
        if key not in seen:
            seen.add(key)
            result.append(e)
    return result


def entity_overlap_score(article_entities, thread_entities, idf=None):
    a_norm = set(e.lower() for e in normalize_entities(article_entities))
    t_norm = set(e.lower() for e in normalize_entities(thread_entities))
    if not a_norm or not t_norm:
        return 0.0
    intersection = a_norm & t_norm
    union = a_norm | t_norm
    if idf:
        inter_weight = sum(idf.get(e, 1.0) for e in intersection)
        union_weight = sum(idf.get(e, 1.0) for e in union)
    else:
        inter_weight = float(len(intersection))
        union_weight = float(len(union))
    return inter_weight / union_weight if union_weight > 0 else 0.0


def compute_time_weighted_centroid(members, reference_date, decay):
    try:
        ref = datetime.strptime(reference_date[:10], '%Y-%m-%d')
    except ValueError:
        ref = datetime.now()
    embeddings, weights = [], []
    for m in members:
        emb = m.get('embedding')
        if emb is None:
            continue
        try:
            days_old = max(0, (ref - datetime.strptime(m['published_at'][:10], '%Y-%m-%d')).days)
        except (ValueError, KeyError):
            days_old = 0
        embeddings.append(emb)
        weights.append(math.exp(-decay * days_old))
    if not embeddings:
        return None
    embs = np.array(embeddings, dtype=np.float32)
    w = np.array(weights, dtype=np.float32)
    centroid = np.average(embs, axis=0, weights=w)
    norm = np.linalg.norm(centroid)
    return centroid / norm if norm > 0 else centroid


def cosine_sim(a, b):
    return float(np.dot(a, b))


def precompute_idf_from_map(entity_map_input):
    """Compute IDF weights from entity_map."""
    entity_doc_count = defaultdict(int)
    total_docs = 0
    for entities in entity_map_input.values():
        unique = set(e.lower() for e in normalize_entities(entities))
        for e in unique:
            entity_doc_count[e] += 1
        total_docs += 1
    if total_docs == 0:
        return {}
    return {
        e: math.log((total_docs + 1) / (count + 1)) + 1
        for e, count in entity_doc_count.items()
    }


# ── Core matching function ──

def simulate_match(articles_day, sim_threads, thread_members, idf_weights,
                   base_threshold, author_threshold, time_penalty,
                   centroid_decay, entity_weight,
                   match_margin=GS_MATCH_MARGIN):
    """Match articles to in-memory threads. Returns (matched_dict, unmatched_list).

    sim_threads: {thread_id: {'title', 'last_seen', 'count', 'centroid'}}
    thread_members: {thread_id: [{'id', 'creator', 'published_at', 'embedding', 'entities'}]}
    """
    matched = {}
    unmatched = []

    for article in articles_day:
        emb = article.get('embedding')
        if emb is None:
            unmatched.append(article)
            continue

        article_date = article.get('published_at', '')[:10]
        article_creator = article.get('creator')
        article_entities = article.get('entities', [])

        best_sim = 0.0
        second_best_sim = 0.0
        best_tid = None

        for tid, tmeta in sim_threads.items():
            members = thread_members.get(tid, [])

            # Time-weighted centroid
            twc = compute_time_weighted_centroid(members, article_date, centroid_decay)
            centroid = twc if twc is not None else tmeta.get('centroid')
            if centroid is None:
                continue

            sim = cosine_sim(emb, centroid)

            # Entity overlap boost
            if entity_weight > 0 and members and article_entities:
                thread_entities_all = []
                for m in members:
                    thread_entities_all.extend(m.get('entities', []))
                if thread_entities_all:
                    overlap = entity_overlap_score(article_entities, thread_entities_all, idf_weights)
                    sim = sim + entity_weight * overlap

            # Time penalty
            days_gap = 0
            if article_date and tmeta.get('last_seen'):
                try:
                    a_dt = datetime.strptime(article_date, '%Y-%m-%d')
                    t_dt = datetime.strptime(tmeta['last_seen'][:10], '%Y-%m-%d')
                    days_gap = abs((a_dt - t_dt).days)
                except ValueError:
                    pass

            effective_threshold = base_threshold + time_penalty * days_gap

            # Hard cap
            if tmeta.get('count', 0) >= GS_HARD_CAP:
                effective_threshold = max(effective_threshold, GS_FROZEN_THRESHOLD)

            # Author boost
            if article_creator and members:
                for m in members:
                    if m.get('creator') != article_creator:
                        continue
                    try:
                        m_dt = datetime.strptime(m['published_at'][:19], '%Y-%m-%dT%H:%M:%S')
                        a_dt_full = datetime.strptime(article.get('published_at', '')[:19], '%Y-%m-%dT%H:%M:%S')
                        if abs((a_dt_full - m_dt).total_seconds()) <= GS_AUTHOR_WINDOW_HOURS * 3600:
                            effective_threshold = min(effective_threshold, author_threshold)
                            break
                    except ValueError:
                        pass

            if sim >= effective_threshold:
                if sim > best_sim:
                    second_best_sim = best_sim
                    best_sim = sim
                    best_tid = tid
                elif sim > second_best_sim:
                    second_best_sim = sim

        # Margin check
        if best_tid and (best_sim - second_best_sim) >= match_margin:
            matched[article['id']] = best_tid
        else:
            unmatched.append(article)

    return matched, unmatched


# ── Full simulation runner (single source of truth) ──

def run_simulation(articles_by_date_input, idf_weights, params, prefix='sim'):
    """Run a full chronological replay with given params.

    Args:
        articles_by_date_input: {date_str: [article_dicts]}
        idf_weights: pre-computed IDF dict
        params: dict with base_threshold, author_threshold, time_penalty,
                centroid_decay, entity_weight
        prefix: thread ID prefix (for disambiguation across runs)

    Returns:
        (matched_map, sim_threads, sim_members)
        matched_map: {article_id: thread_id}
        sim_threads: {thread_id: {title, last_seen, count, centroid}}
        sim_members: {thread_id: [member_dicts]}
    """
    threads = {}
    members = {}
    thread_ctr = 0
    matched_map = {}

    for date_key in sorted(articles_by_date_input.keys()):
        day_articles = articles_by_date_input[date_key]
        day_matched, day_unmatched = simulate_match(
            day_articles, threads, members, idf_weights, **params
        )

        # Process matched articles — update threads
        for aid, tid in day_matched.items():
            matched_map[aid] = tid
            article = next(a for a in day_articles if a['id'] == aid)
            member = {
                'id': aid,
                'creator': article.get('creator'),
                'published_at': article.get('published_at', ''),
                'embedding': article['embedding'],
                'entities': article.get('entities', []),
            }
            members[tid].append(member)
            threads[tid]['last_seen'] = max(threads[tid]['last_seen'], date_key)
            threads[tid]['count'] += 1
            # Running mean centroid update
            n = threads[tid]['count']
            old_c = threads[tid]['centroid']
            new_c = old_c + (article['embedding'] - old_c) / n
            norm = np.linalg.norm(new_c)
            threads[tid]['centroid'] = new_c / norm if norm > 0 else new_c

        # Create single-article threads for unmatched
        for article in day_unmatched:
            tid = f'{prefix}_{thread_ctr:04d}'
            thread_ctr += 1
            threads[tid] = {
                'title': article['title'][:60],
                'last_seen': date_key,
                'count': 1,
                'centroid': article['embedding'].copy(),
            }
            members[tid] = [{
                'id': article['id'],
                'creator': article.get('creator'),
                'published_at': article.get('published_at', ''),
                'embedding': article['embedding'],
                'entities': article.get('entities', []),
            }]
            matched_map[article['id']] = tid

    return matched_map, threads, members


# ── Build sorted article data ──
articles_all = []
for item in sorted(items_rows, key=lambda x: x.get('published_at', '')):
    aid = item['id']
    emb = embedding_map.get(aid)
    if emb is None:
        continue
    articles_all.append({
        'id': aid,
        'title': item['title'],
        'published_at': item.get('published_at', ''),
        'creator': item.get('creator'),
        'entities': entity_map.get(aid, []),
        'summary': summary_map.get(aid, ''),
        'keywords': keywords_map.get(aid, []),
        'embedding': emb,
    })

articles_by_date = defaultdict(list)
for a in articles_all:
    articles_by_date[a['published_at'][:10]].append(a)

print(f'Articles with embeddings: {len(articles_all)}')

# Pre-compute IDF
idf = precompute_idf_from_map(entity_map)
print(f'IDF vocabulary: {len(idf)} entities')

# ── Run low-threshold baseline simulation ──
print('\nRunning low-threshold simulation...')
gs_params = {
    'base_threshold': GS_BASE_THRESHOLD,
    'author_threshold': GS_AUTHOR_THRESHOLD,
    'time_penalty': GS_TIME_PENALTY,
    'centroid_decay': GS_CENTROID_DECAY,
    'entity_weight': GS_ENTITY_WEIGHT,
}
baseline_matched, sim_threads, sim_members = run_simulation(
    articles_by_date, idf, gs_params, prefix='sim'
)

# Singletons = articles without embeddings (not in simulation)
all_article_ids = {r['id'] for r in items_rows}
baseline_singletons = list(all_article_ids - set(baseline_matched.keys()))

print(f'\nBaseline simulation results:')
print(f'  Total threads created: {len(sim_threads)}')
print(f'  Matched articles:      {len(baseline_matched)}')
print(f'  Singletons (no emb):   {len(baseline_singletons)}')
if articles_all:
    rate = len(baseline_matched) / len(articles_all)
    print(f'  Threading rate:        {rate:.1%} (should be > 16.4% baseline)')

# Build structured baseline for Gemini validation (threads with 2+ members)
item_lookup = {r['id']: r for r in items_rows}

baseline_thread_groups = {}
for tid, member_list in sim_members.items():
    if len(member_list) >= 2:
        baseline_thread_groups[tid] = {
            'title': sim_threads[tid]['title'],
            'articles': [
                {
                    'id': m['id'],
                    'title': item_lookup.get(m['id'], {}).get('title', ''),
                    'published_at': m['published_at'],
                    'creator': m.get('creator'),
                    'summary': summary_map.get(m['id'], ''),
                    'keywords': keywords_map.get(m['id'], []),
                }
                for m in member_list
            ]
        }

singleton_ids = [m['id'] for tid, member_list in sim_members.items()
                 if len(member_list) == 1 for m in member_list]

# Build singleton articles list for Cell 7 (Q3)
singleton_articles = [
    {
        'id': aid,
        'title': item_lookup.get(aid, {}).get('title', ''),
        'published_at': item_lookup.get(aid, {}).get('published_at', ''),
        'creator': item_lookup.get(aid, {}).get('creator'),
        'summary': summary_map.get(aid, ''),
        'keywords': keywords_map.get(aid, []),
    }
    for aid in singleton_ids
    if item_lookup.get(aid)
]
singleton_articles.sort(key=lambda a: a.get('published_at', ''))

print(f'\nThreads with 2+ articles: {len(baseline_thread_groups)}')
print(f'Single-article threads (Gemini review targets): {len(singleton_ids)}')

# Save baseline
baseline_path = Path('golden_baseline.json')
with open(baseline_path, 'w') as f:
    json.dump({
        'generated_at': datetime.now().isoformat(),
        'scope_date': SCOPE_DATE,
        'thresholds': {'base': GS_BASE_THRESHOLD, 'author': GS_AUTHOR_THRESHOLD},
        'threads': baseline_thread_groups,
        'singletons': singleton_ids,
    }, f, indent=2)
print(f'\nSaved to {baseline_path.resolve()}')

Articles with embeddings: 2102
IDF vocabulary: 9780 entities

Running low-threshold simulation...

Baseline simulation results:
  Total threads created: 943
  Matched articles:      2102
  Singletons (no emb):   0
  Threading rate:        100.0% (should be > 16.4% baseline)

Threads with 2+ articles: 167
Single-article threads (Gemini review targets): 776

Saved to /Users/youngmincho/Project/araverus/notebooks/golden_baseline.json


In [5]:
# Cell 5: Gemini Q1 — Thread Internal Validation
# Validates each baseline thread: are all articles about the same specific event?

from google import genai
from google.genai import types

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_AI_API_KEY')
assert GEMINI_API_KEY, 'Missing GEMINI_API_KEY'
client = genai.Client(api_key=GEMINI_API_KEY)

Q1_SYSTEM = """You are a news editor validating thread coherence.
Given a list of articles in a thread, determine if they all cover the same SPECIFIC EVENT.

Respond in JSON:
{
  "coherent": true|false,
  "contaminated_ids": ["id1", ...],
  "should_split": true|false,
  "split_groups": [["id1", "id2"], ["id3"]],
  "links": [{"from": "id1", "to": "id2", "type": "topical|causal|analysis"}],
  "confidence": "high|medium|low",
  "notes": "optional brief explanation"
}"""


def _truncate_summary(text, max_sentences=2):
    """Truncate summary to first N sentences, max 120 chars."""
    if not text:
        return ""
    sentences = text.split('. ')
    truncated = '. '.join(sentences[:max_sentences])
    return truncated[:117] + '...' if len(truncated) > 120 else truncated


def build_q1_prompt(thread_title, articles):
    lines = [f'Thread: "{thread_title}"\n\nArticles:']
    for a in articles:
        pub = a.get('published_at', '')[:10]
        creator = a.get('creator', 'unknown')
        line = f'[{a["id"][:8]}] [{pub}] ({creator}) {a["title"]}'
        if a.get('keywords'):
            line += f' [{", ".join(a["keywords"])}]'
        lines.append(line)
        brief = _truncate_summary(a.get('summary'))
        if brief:
            lines.append(f'   > {brief}')
    return '\n'.join(lines)


def call_gemini_q1(thread_title, articles, retries=2):
    prompt = build_q1_prompt(thread_title, articles)
    for attempt in range(retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'{Q1_SYSTEM}\n\n{prompt}',
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    response_mime_type='application/json',
                ),
            )
            return json.loads(resp.text)
        except Exception as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                return {'error': str(e), 'confidence': 'low'}


# Run Q1 validation (batch with rate limiting)
q1_results = {}   # thread_id -> Gemini response
thread_ids_to_validate = list(baseline_thread_groups.keys())
BATCH_DELAY = 1.0  # seconds between calls

# Preview first thread prompt
first_tid = thread_ids_to_validate[0]
first_group = baseline_thread_groups[first_tid]
print('=== Q1 Prompt Preview (first thread) ===')
print(build_q1_prompt(first_group['title'], first_group['articles']))
print('=' * 50)

print(f'\nRunning Q1 validation on {len(thread_ids_to_validate)} threads...')
for i, tid in enumerate(thread_ids_to_validate):
    group = baseline_thread_groups[tid]
    result = call_gemini_q1(group['title'], group['articles'])
    q1_results[tid] = result
    if i % 10 == 0:
        print(f'  Progress: {i+1}/{len(thread_ids_to_validate)}')
    time.sleep(BATCH_DELAY)

# Summary
coherent_count = sum(1 for r in q1_results.values() if r.get('coherent') is True)
split_count = sum(1 for r in q1_results.values() if r.get('should_split') is True)
high_conf = sum(1 for r in q1_results.values() if r.get('confidence') == 'high')
errors = sum(1 for r in q1_results.values() if 'error' in r)

print(f'\nQ1 Results:')
print(f'  Coherent:     {coherent_count}/{len(q1_results)}')
print(f'  Should split: {split_count}')
print(f'  High conf:    {high_conf}')
print(f'  Errors:       {errors}')

# Collect causal links
all_causal_links = []
for tid, result in q1_results.items():
    for link in result.get('links', []):
        if link.get('type') == 'causal':
            all_causal_links.append({**link, 'thread_id': tid})
print(f'  Causal links found: {len(all_causal_links)}')

=== Q1 Prompt Preview (first thread) ===
Thread: "China Private Gauge Signals Weaker Manufacturing Activity"

Articles:
[765e032a] [2025-12-01] (None) China Private Gauge Signals Weaker Manufacturing Activity [China economy, manufacturing PMI, factory activity, economic contraction]
   > China's factory activity showed a slight increase in November but remained in contraction for the eighth month, with ...
[9345dcf4] [2025-12-02] (Paul Hannon) Growth to Slow as Tariffs Bite, but AI Investments May Cushion the Blow [global economy, AI investments, tariffs, interest rate cuts]
   > The global economy is projected to grow at 3% in 2026, down from 3.4% in 2025, with AI investments and rate cuts from...
[1c60bf84] [2025-12-05] (Paul Hannon) Eurozone Economy Posts Stronger Growth on Investment Rebound [Brexit impact, UK economy, slow growth]
   > The crawled article discusses the negative impact of Brexit on the UK economy, focusing on slow growth, weak investme...

Running Q1 validation on 

In [6]:
# Save Q1 results (run after Cell 5 completes)
with open('q1_results.pkl', 'wb') as f:
    pickle.dump(q1_results, f)
print(f'Saved {len(q1_results)} Q1 results to q1_results.pkl')

# To restore if kernel dies:
# with open('q1_results.pkl', 'rb') as f:
#     q1_results = pickle.load(f)

Saved 167 Q1 results to q1_results.pkl


In [7]:
# Cell 6: Gemini Q2 — Thread-Pair Comparison
# For similar thread pairs (centroid similarity > 0.65): merge / parent-child / separate?

Q2_SYSTEM = """You are a news editor. Given two threads, decide their relationship.

Respond in JSON:
{
  "relationship": "merge|parent_child|separate",
  "parent_thread": "thread_id or null (for parent_child)",
  "confidence": "high|medium|low",
  "notes": "optional"
}"""


def build_q2_prompt(tid_a, group_a, tid_b, group_b):
    lines = [f'Thread A ({tid_a[:8]}): "{group_a["title"]}"']
    for a in group_a['articles'][:5]:
        lines.append(f'  - [{a["id"][:8]}] {a["title"]}')
    lines.append(f'\nThread B ({tid_b[:8]}): "{group_b["title"]}"')
    for a in group_b['articles'][:5]:
        lines.append(f'  - [{a["id"][:8]}] {a["title"]}')
    return '\n'.join(lines)


# Find similar thread pairs by centroid similarity
PAIR_SIM_THRESHOLD = 0.65
thread_list = list(baseline_thread_groups.items())
similar_pairs = []

for i in range(len(thread_list)):
    tid_a, group_a = thread_list[i]
    c_a = sim_threads[tid_a]['centroid']
    for j in range(i + 1, len(thread_list)):
        tid_b, group_b = thread_list[j]
        c_b = sim_threads[tid_b]['centroid']
        sim = cosine_sim(c_a, c_b)
        if sim >= PAIR_SIM_THRESHOLD:
            similar_pairs.append((tid_a, tid_b, sim))

similar_pairs.sort(key=lambda x: x[2], reverse=True)
print(f'Similar thread pairs (sim > {PAIR_SIM_THRESHOLD}): {len(similar_pairs)}')

# Cap at top 50 pairs to control API cost
pairs_to_check = similar_pairs[:50]
print(f'Running Q2 on top {len(pairs_to_check)} pairs...')

q2_results = {}  # (tid_a, tid_b) -> result
for k, (tid_a, tid_b, sim) in enumerate(pairs_to_check):
    group_a = baseline_thread_groups[tid_a]
    group_b = baseline_thread_groups[tid_b]
    prompt = build_q2_prompt(tid_a, group_a, tid_b, group_b)
    for attempt in range(3):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'{Q2_SYSTEM}\n\n{prompt}',
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    response_mime_type='application/json',
                ),
            )
            q2_results[(tid_a, tid_b)] = {**json.loads(resp.text), 'sim': sim}
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                q2_results[(tid_a, tid_b)] = {'error': str(e), 'sim': sim}
    if k % 10 == 0:
        print(f'  Progress: {k+1}/{len(pairs_to_check)}')
    time.sleep(BATCH_DELAY)

# Summary
rel_counts = defaultdict(int)
for r in q2_results.values():
    rel_counts[r.get('relationship', 'error')] += 1

print('\nQ2 Results:')
for rel, cnt in sorted(rel_counts.items()):
    print(f'  {rel}: {cnt}')

Similar thread pairs (sim > 0.65): 2722
Running Q2 on top 50 pairs...
  Progress: 1/50
  Progress: 11/50
  Progress: 21/50
  Progress: 31/50
  Progress: 41/50

Q2 Results:
  merge: 6
  parent_child: 2
  separate: 42


In [8]:
# Cell 7: Gemini Q3 — Singleton Validation
# Batch singletons by date proximity so Gemini can find temporal clusters.

Q3_SYSTEM = """You are a news editor reviewing standalone articles.
Given a list of articles, identify groups that should form new threads,
and articles that should join an existing thread (provided as context).

IMPORTANT: Articles about the same developing story within a few days of each other
should be grouped together.

Respond in JSON:
{
  "new_groups": [
    {"title": "Thread title", "article_ids": ["id1", "id2"]}
  ],
  "join_existing": [
    {"article_id": "id", "thread_title": "existing thread title"}
  ],
  "truly_standalone": ["id3", "id4"],
  "confidence": "high|medium|low"
}"""


def build_q3_prompt(singleton_articles, existing_thread_titles):
    lines = ['Standalone articles to review:']
    for a in singleton_articles:
        pub = a.get('published_at', '')[:10]
        lines.append(f'[{a["id"][:8]}] [{pub}] {a.get("title", "")}')
    if existing_thread_titles:
        lines.append('\nExisting threads for reference:')
        for title in existing_thread_titles[:20]:
            lines.append(f'  - {title}')
    return '\n'.join(lines)


# Prepare singleton articles — sorted by date
singleton_articles = [
    {
        'id': aid,
        'title': item_lookup.get(aid, {}).get('title', ''),
        'published_at': item_lookup.get(aid, {}).get('published_at', ''),
    }
    for aid in singleton_ids
    if item_lookup.get(aid)
]
singleton_articles.sort(key=lambda a: a.get('published_at', ''))

existing_titles = [g['title'] for g in baseline_thread_groups.values()][:30]

# Batch by date windows (7-day sliding window, max 35 per batch)
# This ensures articles close in time are in the same batch
BATCH_SIZE_Q3 = 35
q3_results = []

print(f'Running Q3 on {len(singleton_articles)} singletons (date-sorted, batches of {BATCH_SIZE_Q3})...')
for i in range(0, len(singleton_articles), BATCH_SIZE_Q3):
    batch = singleton_articles[i:i + BATCH_SIZE_Q3]
    date_range = f'{batch[0]["published_at"][:10]} → {batch[-1]["published_at"][:10]}'
    prompt = build_q3_prompt(batch, existing_titles)
    for attempt in range(3):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'{Q3_SYSTEM}\n\n{prompt}',
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    response_mime_type='application/json',
                ),
            )
            q3_results.append(json.loads(resp.text))
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                q3_results.append({'error': str(e)})
    print(f'  Batch {i // BATCH_SIZE_Q3 + 1}: {len(batch)} articles ({date_range})')
    time.sleep(BATCH_DELAY)

# Summary
new_groups_total = sum(len(r.get('new_groups', [])) for r in q3_results)
join_total = sum(len(r.get('join_existing', [])) for r in q3_results)
standalone_total = sum(len(r.get('truly_standalone', [])) for r in q3_results)

print(f'\nQ3 Results:')
print(f'  New groups suggested:   {new_groups_total}')
print(f'  Join existing:          {join_total}')
print(f'  Truly standalone:       {standalone_total}')

Running Q3 on 776 singletons (date-sorted, batches of 35)...
  Batch 1: 35 articles (2025-12-19 → 2026-01-16)
  Batch 2: 35 articles (2026-01-16 → 2026-01-20)
  Batch 3: 35 articles (2026-01-20 → 2026-01-27)
  Batch 4: 35 articles (2026-01-27 → 2026-01-29)
  Batch 5: 35 articles (2026-01-29 → 2026-01-30)
  Batch 6: 35 articles (2026-01-30 → 2026-02-02)
  Batch 7: 35 articles (2026-02-02 → 2026-02-03)
  Batch 8: 35 articles (2026-02-03 → 2026-02-04)
  Batch 9: 35 articles (2026-02-04 → 2026-02-06)
  Batch 10: 35 articles (2026-02-06 → 2026-02-07)
  Batch 11: 35 articles (2026-02-07 → 2026-02-10)
  Batch 12: 35 articles (2026-02-10 → 2026-02-11)
  Batch 13: 35 articles (2026-02-11 → 2026-02-12)
  Batch 14: 35 articles (2026-02-12 → 2026-02-13)
  Batch 15: 35 articles (2026-02-13 → 2026-02-16)
  Batch 16: 35 articles (2026-02-16 → 2026-02-17)
  Batch 17: 35 articles (2026-02-18 → 2026-02-19)
  Batch 18: 35 articles (2026-02-19 → 2026-02-21)
  Batch 19: 35 articles (2026-02-21 → 2026-02-24

In [9]:
# Cell 7: Gemini Q3 — Singleton Validation
# Batch singletons by date proximity so Gemini can find temporal clusters.

Q3_SYSTEM = """You are a news editor reviewing standalone articles.
Given a list of articles, identify groups that should form new threads,
and articles that should join an existing thread (provided as context).

IMPORTANT: Articles about the same developing story within a few days of each other
should be grouped together.

Respond in JSON:
{
  "new_groups": [
    {"title": "Thread title", "article_ids": ["id1", "id2"]}
  ],
  "join_existing": [
    {"article_id": "id", "thread_title": "existing thread title"}
  ],
  "truly_standalone": ["id3", "id4"],
  "confidence": "high|medium|low"
}"""


def build_q3_prompt(singleton_articles_batch, existing_thread_titles):
    lines = ['Standalone articles to review:']
    for a in singleton_articles_batch:
        pub = a.get('published_at', '')[:10]
        creator = a.get('creator', 'unknown')
        line = f'[{a["id"][:8]}] [{pub}] ({creator}) {a.get("title", "")}'
        if a.get('keywords'):
            line += f' [{", ".join(a["keywords"])}]'
        lines.append(line)
        brief = _truncate_summary(a.get('summary'))
        if brief:
            lines.append(f'   > {brief}')
    if existing_thread_titles:
        lines.append('\nExisting threads for reference:')
        for title in existing_thread_titles[:20]:
            lines.append(f'  - {title}')
    return '\n'.join(lines)


existing_titles = [g['title'] for g in baseline_thread_groups.values()][:30]

# Batch by date windows (7-day sliding window, max 35 per batch)
BATCH_SIZE_Q3 = 35
q3_results = []

print(f'Running Q3 on {len(singleton_articles)} singletons (date-sorted, batches of {BATCH_SIZE_Q3})...')
for i in range(0, len(singleton_articles), BATCH_SIZE_Q3):
    batch = singleton_articles[i:i + BATCH_SIZE_Q3]
    date_range = f'{batch[0]["published_at"][:10]} → {batch[-1]["published_at"][:10]}'
    prompt = build_q3_prompt(batch, existing_titles)
    for attempt in range(3):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'{Q3_SYSTEM}\n\n{prompt}',
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    response_mime_type='application/json',
                ),
            )
            q3_results.append(json.loads(resp.text))
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                q3_results.append({'error': str(e)})
    print(f'  Batch {i // BATCH_SIZE_Q3 + 1}: {len(batch)} articles ({date_range})')
    time.sleep(BATCH_DELAY)

# Summary
new_groups_total = sum(len(r.get('new_groups', [])) for r in q3_results)
join_total = sum(len(r.get('join_existing', [])) for r in q3_results)
standalone_total = sum(len(r.get('truly_standalone', [])) for r in q3_results)

print(f'\nQ3 Results:')
print(f'  New groups suggested:   {new_groups_total}')
print(f'  Join existing:          {join_total}')
print(f'  Truly standalone:       {standalone_total}')

Running Q3 on 776 singletons (date-sorted, batches of 35)...
  Batch 1: 35 articles (2025-12-19 → 2026-01-16)
  Batch 2: 35 articles (2026-01-16 → 2026-01-20)
  Batch 3: 35 articles (2026-01-20 → 2026-01-27)
  Batch 4: 35 articles (2026-01-27 → 2026-01-29)
  Batch 5: 35 articles (2026-01-29 → 2026-01-30)
  Batch 6: 35 articles (2026-01-30 → 2026-02-02)
  Batch 7: 35 articles (2026-02-02 → 2026-02-03)
  Batch 8: 35 articles (2026-02-03 → 2026-02-04)
  Batch 9: 35 articles (2026-02-04 → 2026-02-06)
  Batch 10: 35 articles (2026-02-06 → 2026-02-07)
  Batch 11: 35 articles (2026-02-07 → 2026-02-10)
  Batch 12: 35 articles (2026-02-10 → 2026-02-11)
  Batch 13: 35 articles (2026-02-11 → 2026-02-12)
  Batch 14: 35 articles (2026-02-12 → 2026-02-13)
  Batch 15: 35 articles (2026-02-13 → 2026-02-16)
  Batch 16: 35 articles (2026-02-16 → 2026-02-17)
  Batch 17: 35 articles (2026-02-18 → 2026-02-19)
  Batch 18: 35 articles (2026-02-19 → 2026-02-21)
  Batch 19: 35 articles (2026-02-21 → 2026-02-24

In [10]:
# Cell 9: Evaluation Harness

def evaluate_threading(system_threads_input, golden_dataset):
    """
    Evaluate a threading assignment against the golden dataset.

    Args:
        system_threads_input: dict of {thread_id: [article_id, ...]}
                              OR {article_id: thread_id} (article→thread map)
        golden_dataset: dict with 'threads' list and 'singletons' list

    Returns:
        dict with:
          contamination:  % of system threads mixing golden threads (lower is better)
          fragmentation:  % of golden threads split across system threads (lower is better)
          causal_recall:  % of golden causal links in same system thread (higher is better)
          composite:      weighted score (see formula below)
    """
    golden_threads = golden_dataset.get('threads', [])

    # Normalize system_threads to article->thread_id map
    if system_threads_input and isinstance(next(iter(system_threads_input.values())), list):
        art_to_sys = {}
        for tid, articles in system_threads_input.items():
            for aid in articles:
                art_to_sys[aid] = tid
    else:
        art_to_sys = dict(system_threads_input)  # already article->thread

    # Build golden article->golden_thread map
    art_to_golden = {}
    for i, gt in enumerate(golden_threads):
        gid = f'golden_{i}'
        for aid in gt['articles']:
            art_to_golden[aid] = gid

    # ── Contamination: system threads mixing different golden threads ──
    sys_thread_to_goldens = defaultdict(set)
    for aid, sys_tid in art_to_sys.items():
        g_tid = art_to_golden.get(aid)
        if g_tid:
            sys_thread_to_goldens[sys_tid].add(g_tid)

    if sys_thread_to_goldens:
        contaminated_sys = sum(1 for goldens in sys_thread_to_goldens.values() if len(goldens) > 1)
        contamination = contaminated_sys / len(sys_thread_to_goldens)
    else:
        contamination = 0.0

    # ── Fragmentation: golden threads split across system threads ──
    fragmented_golden = 0
    for gt in golden_threads:
        sys_threads_for_golden = set(
            art_to_sys.get(aid)
            for aid in gt['articles']
            if art_to_sys.get(aid)
        )
        if len(sys_threads_for_golden) > 1:
            fragmented_golden += 1

    fragmentation = fragmented_golden / len(golden_threads) if golden_threads else 0.0

    # ── Causal recall: golden causal links in same system thread ──
    causal_links = [
        lnk for gt in golden_threads
        for lnk in gt.get('links', [])
        if lnk.get('type') == 'causal'
    ]
    n_causal = len(causal_links)
    if n_causal > 0:
        recalled = 0
        for lnk in causal_links:
            from_id = lnk['from']
            to_id = lnk['to']
            # Handle 8-char prefix IDs from Gemini
            from_sys = art_to_sys.get(from_id) or next(
                (art_to_sys[k] for k in art_to_sys if k.startswith(from_id)), None
            )
            to_sys = art_to_sys.get(to_id) or next(
                (art_to_sys[k] for k in art_to_sys if k.startswith(to_id)), None
            )
            if from_sys and to_sys and from_sys == to_sys:
                recalled += 1
        causal_recall = recalled / n_causal
    else:
        causal_recall = 0.0  # No causal links → 0 (not 1.0 — avoid inflating composite)

    # ── Composite score ──
    # When no causal links exist, redistribute weight to contamination + fragmentation
    if n_causal > 0:
        composite = (1 - contamination) * 0.3 + (1 - fragmentation) * 0.3 + causal_recall * 0.4
    else:
        composite = (1 - contamination) * 0.5 + (1 - fragmentation) * 0.5

    # ── Threading rate: fraction of golden articles assigned to multi-article threads ──
    golden_article_ids = set(art_to_golden.keys())
    multi_article_sys_threads = set()
    sys_thread_article_counts = defaultdict(int)
    for aid in golden_article_ids:
        sys_tid = art_to_sys.get(aid)
        if sys_tid:
            sys_thread_article_counts[sys_tid] += 1
    for sys_tid, count in sys_thread_article_counts.items():
        if count >= 2:
            multi_article_sys_threads.add(sys_tid)
    golden_in_multi = sum(
        1 for aid in golden_article_ids
        if art_to_sys.get(aid) in multi_article_sys_threads
    )
    threading_rate = golden_in_multi / len(golden_article_ids) if golden_article_ids else 0.0

    return {
        'contamination': contamination,
        'fragmentation': fragmentation,
        'causal_recall': causal_recall,
        'composite': composite,
        'threading_rate': threading_rate,
        'n_system_threads': len(sys_thread_to_goldens),
        'n_golden_threads': len(golden_threads),
        'n_causal_links': n_causal,
    }


# Load golden dataset
with open('golden_dataset.json') as f:
    golden_dataset = json.load(f)

# Evaluate current DB system threads
print('=== Baseline System Score (current DB state) ===')
baseline_score = evaluate_threading(dict(system_threads), golden_dataset)
for k, v in baseline_score.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print('\n=== Low-threshold Simulation Score ===')
sim_score = evaluate_threading(dict(baseline_matched), golden_dataset)
for k, v in sim_score.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

=== Baseline System Score (current DB state) ===
  contamination: 0.2821
  fragmentation: 0.1940
  causal_recall: 0.1600
  composite: 0.5212
  threading_rate: 0.2037
  n_system_threads: 78
  n_golden_threads: 134
  n_causal_links: 25

=== Low-threshold Simulation Score ===
  contamination: 0.2245
  fragmentation: 0.0000
  causal_recall: 1.0000
  composite: 0.9327
  threading_rate: 1.0000
  n_system_threads: 98
  n_golden_threads: 134
  n_causal_links: 25


In [ ]:
# Cell 10: Grid Search Simulator
# Sweep parameter combinations (no DB calls — all in-memory)
# Uses run_simulation() from Cell 4 — single source of truth.

GRID = {
    'base_threshold':   [0.60, 0.65, 0.68, 0.70, 0.73, 0.75, 0.78, 0.80],
    'author_threshold': [0.50, 0.55, 0.60, 0.65],
    'time_penalty':     [0.005, 0.01, 0.015, 0.02],
    'centroid_decay':   [0.05, 0.08, 0.10, 0.15],
    'entity_weight':    [0.02, 0.04, 0.06],
}

total_combos = 1
for v in GRID.values():
    total_combos *= len(v)
print(f'Total combinations: {total_combos}')

# ── Validation step: current constants should reproduce DB-like results ──
print('\nValidation: simulating with current production constants...')
PROD_PARAMS = {
    'base_threshold': 0.73,
    'author_threshold': 0.60,
    'time_penalty': 0.01,
    'centroid_decay': 0.10,
    'entity_weight': 0.04,
}

prod_matched, prod_threads, prod_members = run_simulation(
    articles_by_date, idf, PROD_PARAMS, prefix='prod'
)

prod_multi = sum(1 for tid, m in prod_members.items() if len(m) >= 2)
db_multi = sum(1 for tid, arts in system_threads.items() if len(arts) >= 2)
print(f'Production sim: {len(prod_matched)} matched, {prod_multi} multi-article threads')
print(f'Actual DB:      {len(article_thread_map)} matched, {db_multi} multi-article threads')
mismatch_pct = abs(prod_multi - db_multi) / max(db_multi, 1)
print(f'Multi-thread mismatch: {mismatch_pct:.1%}')
if mismatch_pct > 0.30:
    print('WARNING: >30% mismatch — simulator may not accurately reflect pipeline.')
    print('  Expected: simulation only does centroid matching (no LLM grouping).')
    print('  DB includes LLM-grouped threads. Mismatch is normal but tracked.')

# ── Grid Search ──
print(f'\nStarting grid search ({total_combos} combinations)...')
grid_results = []

keys = list(GRID.keys())
values = list(GRID.values())

start_time = time.time()
for i, combo in enumerate(itertools.product(*values)):
    params = dict(zip(keys, combo))

    gs_matched, _, _ = run_simulation(
        articles_by_date, idf, params, prefix='gs'
    )

    scores = evaluate_threading(gs_matched, golden_dataset)
    grid_results.append({**params, **scores})

    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_time
        eta = elapsed / (i + 1) * (total_combos - i - 1)
        print(f'  {i+1}/{total_combos} ({elapsed:.0f}s elapsed, ETA {eta:.0f}s)')

elapsed_total = time.time() - start_time
print(f'\nGrid search complete in {elapsed_total:.1f}s ({elapsed_total/total_combos*1000:.1f}ms/combo)')

# Save results
with open('gridsearch_results.pkl', 'wb') as f:
    pickle.dump(grid_results, f)
print(f'Saved {len(grid_results)} results to gridsearch_results.pkl')

Total combinations: 1536

Validation: simulating with current production constants...
Production sim: 2102 matched, 188 multi-article threads
Actual DB:      374 matched, 102 multi-article threads
Multi-thread mismatch: 84.3%
  Expected: simulation only does centroid matching (no LLM grouping).
  DB includes LLM-grouped threads. Mismatch is normal but tracked.

Starting grid search (1536 combinations)...
  100/1536 (24600s elapsed, ETA 353250s)
  200/1536 (46731s elapsed, ETA 312161s)


In [ ]:
# Cell 11: Grid Search Results Analysis

import matplotlib.pyplot as plt
import seaborn as sns

# Load results
with open('gridsearch_results.pkl', 'rb') as f:
    grid_results = pickle.load(f)

df_grid = pd.DataFrame(grid_results)
df_grid = df_grid.sort_values('composite', ascending=False).reset_index(drop=True)

print('=== Top 10 Combinations by Composite Score ===')
display(df_grid.head(10)[[
    'base_threshold', 'author_threshold', 'time_penalty',
    'centroid_decay', 'entity_weight',
    'contamination', 'fragmentation', 'causal_recall', 'threading_rate', 'composite'
]].style.format({
    'base_threshold': '{:.2f}', 'author_threshold': '{:.2f}',
    'time_penalty': '{:.3f}', 'centroid_decay': '{:.2f}', 'entity_weight': '{:.2f}',
    'contamination': '{:.4f}', 'fragmentation': '{:.4f}',
    'causal_recall': '{:.4f}', 'threading_rate': '{:.1%}', 'composite': '{:.4f}',
}))

# Current production params score
prod_row = df_grid[
    (df_grid['base_threshold'] == 0.73) &
    (df_grid['author_threshold'] == 0.60) &
    (df_grid['time_penalty'] == 0.01) &
    (df_grid['centroid_decay'] == 0.10) &
    (df_grid['entity_weight'] == 0.04)
]
if not prod_row.empty:
    prod_rank = prod_row.index[0] + 1
    prod_composite = prod_row['composite'].iloc[0]
    best_composite = df_grid['composite'].iloc[0]
    print(f'\nProduction params rank: {prod_rank}/{len(df_grid)}')
    print(f'Production composite:   {prod_composite:.4f}')
    print(f'Best composite:         {best_composite:.4f}')
    print(f'Improvement:            {(best_composite - prod_composite):.4f} ({(best_composite/prod_composite - 1):.1%})')

# Heatmap 1: base_threshold vs centroid_decay
pivot1 = df_grid.groupby(['base_threshold', 'centroid_decay'])['composite'].max().unstack()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pivot1, ax=ax1, annot=True, fmt='.3f', cmap='YlOrRd')
ax1.set_title('Max Composite: base_threshold vs centroid_decay')

# Heatmap 2: entity_weight vs author_threshold
pivot2 = df_grid.groupby(['entity_weight', 'author_threshold'])['composite'].max().unstack()
sns.heatmap(pivot2, ax=ax2, annot=True, fmt='.3f', cmap='YlOrRd')
ax2.set_title('Max Composite: entity_weight vs author_threshold')

plt.tight_layout()
plt.savefig('gridsearch_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

# Distribution of individual metrics
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, metric, color in [
    (axes[0], 'contamination', 'salmon'),
    (axes[1], 'fragmentation', 'steelblue'),
    (axes[2], 'causal_recall', 'green'),
    (axes[3], 'threading_rate', 'purple'),
]:
    ax.hist(df_grid[metric], bins=40, color=color, alpha=0.7)
    ax.set_title(metric)
    ax.set_xlabel('Score')
plt.tight_layout()
plt.show()

# Parameter sensitivity: mean composite by each param value
print('\n=== Parameter Sensitivity ===')
for param in GRID.keys():
    by_param = df_grid.groupby(param)['composite'].mean().sort_index()
    best_val = by_param.idxmax()
    print(f'{param}: best={best_val}, range=[{by_param.min():.4f}, {by_param.max():.4f}]')

In [ ]:
# Cell 12: Apply Best Params + Re-thread Plan

best_row = df_grid.iloc[0]

print('=== Best Combination Found ===')
print()
print('# Paste into scripts/7_embed_and_thread.py:')
print(f'THREAD_BASE_THRESHOLD = {best_row["base_threshold"]}')
print(f'AUTHOR_BOOST_THRESHOLD = {best_row["author_threshold"]}')
print(f'THREAD_TIME_PENALTY = {best_row["time_penalty"]}')
print(f'CENTROID_DECAY = {best_row["centroid_decay"]}')
print(f'ENTITY_WEIGHT = {best_row["entity_weight"]}')

print()
print('=== Expected Improvement ===')
if not prod_row.empty:
    for metric in ['contamination', 'fragmentation', 'causal_recall', 'threading_rate', 'composite']:
        before = prod_row[metric].iloc[0]
        after = best_row[metric]
        # For contamination/fragmentation, lower is better
        if metric in ('contamination', 'fragmentation'):
            arrow = '↓ (better)' if after < before else '↑ (worse)'
        else:
            arrow = '↑ (better)' if after > before else '↓ (worse)'
        fmt = '{:.1%}' if metric == 'threading_rate' else '{:.4f}'
        print(f'  {metric:20s}: {fmt.format(before)} → {fmt.format(after)} {arrow}')

print()
print('=== Action Plan ===')
print('1. Review best params above — are they sensible?')
print('2. Update scripts/7_embed_and_thread.py constants')
print('3. Run re-backfill:')
print('   cd scripts')
print('   python 7_embed_and_thread.py --backfill-by-date --start-date 2026-01-01')
print('4. Verify: compare new threading rate vs old 16.4%')
print('5. Update docs/1.2-news-threading.md constants table')

# Summary & Next Steps

## Results (fill after running)

| Metric | Baseline (DB) | Low-threshold sim | Best grid search |
|--------|-------------|-------------------|------------------|
| contamination | - | - | - |
| fragmentation | - | - | - |
| causal_recall | - | - | - |
| composite | - | - | - |
| threading rate | 16.4% | - | - |

## Phase 5 (next)
- Finalize best params → apply to `scripts/7_embed_and_thread.py`
- Run re-backfill with new constants
- Update `docs/1.2-news-threading.md` constants table

## Phase 6 (parent threads)
- Parent thread concept: group related threads under a story arc
- Example: "Fed Rate Decisions Q1" as parent, individual decision threads as children
- Requires schema changes + frontend updates
- See `docs/2.1-thread-stories.md` for hierarchy design

## Files
| File | Contents |
|------|----------|
| `notebooks/threading_gridsearch.ipynb` | This notebook |
| `notebooks/golden_baseline.json` | Low-threshold baseline for Gemini validation |
| `notebooks/golden_dataset.json` | Gemini-validated ground truth |
| `notebooks/gridsearch_results.pkl` | All 1,152 grid search results |
| `notebooks/gridsearch_heatmaps.png` | Heatmap visualizations |